In [6]:
import os
import torch
import soundfile as sf
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps

In [8]:
len(os.listdir("/home/isol/work/data/raw/temp_cut"))

61884

In [7]:
# =====================
# 설정
# =====================
input_folder = "/home/isol/work/data/raw/temp_cut"      # 1초 wav 파일들이 있는 폴더
speech_folder = "/home/isol/work/data/raw/vad_speech"   # VAD가 감지한 경우
silence_folder = "/home/isol/work/data/raw/vad_silence" # VAD가 없을 경우
os.makedirs(speech_folder, exist_ok=True)
os.makedirs(silence_folder, exist_ok=True)

# Silero VAD 모델 로드
model = load_silero_vad()   # 로컬 모델 로드
model.to(torch.device('cuda'))  # GPU 사용

RecursiveScriptModule(
  original_name=VADRNNJITMerge
  (_model): RecursiveScriptModule(
    original_name=VADRNNJIT
    (stft): RecursiveScriptModule(
      original_name=STFT
      (padding): RecursiveScriptModule(original_name=ReflectionPad1d)
    )
    (encoder): RecursiveScriptModule(
      original_name=Sequential
      (0): RecursiveScriptModule(
        original_name=SileroVadBlock
        (se): RecursiveScriptModule(original_name=Identity)
        (activation): RecursiveScriptModule(original_name=ReLU)
        (reparam_conv): RecursiveScriptModule(original_name=Conv1d)
      )
      (1): RecursiveScriptModule(
        original_name=SileroVadBlock
        (se): RecursiveScriptModule(original_name=Identity)
        (activation): RecursiveScriptModule(original_name=ReLU)
        (reparam_conv): RecursiveScriptModule(original_name=Conv1d)
      )
      (2): RecursiveScriptModule(
        original_name=SileroVadBlock
        (se): RecursiveScriptModule(original_name=Identity)
     

In [9]:
for file in os.listdir(input_folder):
    if not file.endswith(".wav"):
        continue

    file_path = os.path.join(input_folder, file)
    audio, sr = sf.read(file_path)
    audio_tensor = torch.tensor(audio, dtype=torch.float32).cuda()

    # speech timestamp 확인
    speech_timestamps = get_speech_timestamps(
        audio_tensor,
        model,
        sampling_rate=sr,
        threshold=0.4,  # 모델이 출력한 speech 확률이 이 값 이상이면 "음성"으로 판단
        min_speech_duration_ms=100, # 이 길이보다 짧은 음성 구간은 무시
        min_silence_duration_ms=30,    # 이 시간 이상 silence가 이어져야 구간을 끊음
        speech_pad_ms=30    # 검출된 음성 구간 앞뒤로 이만큼 padding을 붙여줌
    )

    # 한 개라도 speech segment 있으면 unknown
    if len(speech_timestamps) > 0:
        dest_folder = speech_folder
    else:
        dest_folder = silence_folder

    os.rename(file_path, os.path.join(dest_folder, file))

In [10]:
len(os.listdir("/home/isol/work/data/raw/vad_silence")), len(os.listdir("/home/isol/work/data/raw/vad_speech"))

(15660, 46224)